# Verify Persistence

Confirm that `PersistenceActor` writes to PostgreSQL are complete,
consistent, and correctly typed. Run this after a paper trading session
(or after any changes to `PersistenceActor` or the schema).

Checks:
1. Run lifecycle — `strategy_runs` row exists with `stopped_at` populated
2. Fill completeness — no gaps, no NULLs in required fields, prices stored as NUMERIC
3. Position completeness — every closed position has fills, PnL is populated
4. Fill ↔ position cross-reference — fill counts align with position count
5. Account snapshots — periodic, monotonic timestamps, no missing fields
6. Data type verification — financial columns are NUMERIC, not float

**Prerequisites:** A completed paper trading run (graceful shutdown via
Ctrl+C or `docker compose stop trader`). Infrastructure running
(`docker compose up -d postgres`).

See `VERIFICATION.md` for the full verification plan.

In [ ]:
# ── Cell 1: Config + connect ──────────────────────────────────────

import pandas as pd
import sqlalchemy as sa

from src.config.settings import get_settings

settings = get_settings()
engine = sa.create_engine(settings.postgres_dsn.replace("postgresql://", "postgresql+psycopg2://"))

# Pick a specific run_id, or leave as None to use the most recent run.
RUN_ID = None

print(f"Connected to: {settings.postgres_dsn.split('@')[1]}")

In [ ]:
# ── Cell 2: List recent runs ──────────────────────────────────────

runs_df = pd.read_sql("""
    SELECT id, strategy_id, instrument_id, run_mode,
           started_at, stopped_at, config
    FROM strategy_runs
    ORDER BY started_at DESC
    LIMIT 10
""", engine)

print(f"Recent runs ({len(runs_df)}):")
display(runs_df[["id", "strategy_id", "instrument_id", "run_mode", "started_at", "stopped_at"]])

# Select the run to verify
if RUN_ID is None:
    RUN_ID = str(runs_df.iloc[0]["id"])
    print(f"\nUsing most recent run: {RUN_ID}")
else:
    print(f"\nUsing specified run: {RUN_ID}")

run_row = runs_df[runs_df["id"].astype(str) == RUN_ID].iloc[0]
print(f"Strategy:   {run_row['strategy_id']}")
print(f"Instrument: {run_row['instrument_id']}")
print(f"Mode:       {run_row['run_mode']}")
print(f"Started:    {run_row['started_at']}")
print(f"Stopped:    {run_row['stopped_at']}")

In [ ]:
# ── Cell 3: Run lifecycle ─────────────────────────────────────────
#
# A healthy run has:
# - stopped_at populated (graceful shutdown)
# - started_at < stopped_at
# - config is valid JSON with expected keys

lifecycle_issues = []

# Graceful shutdown
if pd.isna(run_row["stopped_at"]):
    lifecycle_issues.append("stopped_at is NULL — ungraceful shutdown or still running")
    print("❌ stopped_at is NULL")
    print("   Either the run is still active, or it crashed without triggering")
    print("   the SIGTERM handler. Check docker logs or process status.")
else:
    duration = run_row["stopped_at"] - run_row["started_at"]
    print(f"✅ Graceful shutdown — ran for {duration}")

# Config
config = run_row["config"]
if isinstance(config, dict) and len(config) > 0:
    print(f"✅ Config present: {config}")
else:
    lifecycle_issues.append("config is empty or invalid")
    print(f"⚠️  Config: {config}")

if not lifecycle_issues:
    print("\n✅ Run lifecycle OK")
else:
    print(f"\n⚠️  {len(lifecycle_issues)} lifecycle issue(s)")

In [ ]:
# ── Cell 4: Order fills ───────────────────────────────────────────

fills_df = pd.read_sql("""
    SELECT ts, strategy_id, instrument_id, client_order_id,
           venue_order_id, trade_id, order_side,
           last_qty, last_px, commission, commission_currency,
           liquidity_side
    FROM order_fills
    WHERE run_id = %(run_id)s
    ORDER BY ts
""", engine, params={"run_id": RUN_ID})

fill_issues = []

print(f"Total fills: {len(fills_df)}")

if fills_df.empty:
    print("⚠️  No fills found — either no trades occurred or PersistenceActor didn't write")
else:
    print(f"Date range: {fills_df['ts'].min()} → {fills_df['ts'].max()}")
    print(f"Buy fills:  {(fills_df['order_side'] == 'BUY').sum()}")
    print(f"Sell fills: {(fills_df['order_side'] == 'SELL').sum()}")

    # Required fields should never be NULL
    required = ["ts", "strategy_id", "instrument_id", "client_order_id",
                "order_side", "last_qty", "last_px"]
    for col in required:
        nulls = fills_df[col].isna().sum()
        if nulls > 0:
            fill_issues.append(f"{col} has {nulls} NULLs")
            print(f"  ❌ {col}: {nulls} NULL values")

    # Prices and quantities must be positive
    bad_px = (fills_df["last_px"].astype(float) <= 0).sum()
    bad_qty = (fills_df["last_qty"].astype(float) <= 0).sum()
    if bad_px > 0:
        fill_issues.append(f"{bad_px} fills with zero/negative price")
        print(f"  ❌ {bad_px} fills with zero/negative price")
    if bad_qty > 0:
        fill_issues.append(f"{bad_qty} fills with zero/negative quantity")
        print(f"  ❌ {bad_qty} fills with zero/negative quantity")

    # order_side must be BUY or SELL
    valid_sides = {"BUY", "SELL"}
    bad_sides = fills_df[~fills_df["order_side"].isin(valid_sides)]
    if len(bad_sides) > 0:
        fill_issues.append(f"{len(bad_sides)} fills with invalid order_side")
        print(f"  ❌ Invalid order_side values: {bad_sides['order_side'].unique()}")

    # Timestamps should be monotonically non-decreasing
    ts_sorted = fills_df["ts"].is_monotonic_increasing
    if not ts_sorted:
        fill_issues.append("Fill timestamps not monotonically increasing")
        print("  ⚠️  Fill timestamps are not monotonically increasing")

    # Duplicate client_order_id check (same fill written twice)
    dupes = fills_df["client_order_id"].duplicated().sum()
    if dupes > 0:
        fill_issues.append(f"{dupes} duplicate client_order_ids")
        print(f"  ⚠️  {dupes} duplicate client_order_id values")

    if not fill_issues:
        print("\n✅ Order fills OK")
    else:
        print(f"\n⚠️  {len(fill_issues)} fill issue(s)")

    # Show sample
    print("\nSample fills:")
    display(fills_df.head(5))

In [ ]:
# ── Cell 5: Positions ─────────────────────────────────────────────

positions_df = pd.read_sql("""
    SELECT ts_opened, ts_closed, strategy_id, instrument_id,
           position_id, entry_side, peak_qty,
           avg_px_open, avg_px_close,
           realized_pnl, realized_return, currency, duration_ns
    FROM positions
    WHERE run_id = %(run_id)s
    ORDER BY ts_closed
""", engine, params={"run_id": RUN_ID})

pos_issues = []

print(f"Total closed positions: {len(positions_df)}")

if positions_df.empty:
    print("⚠️  No closed positions — run may be too short for a round-trip trade")
else:
    print(f"Date range: {positions_df['ts_opened'].min()} → {positions_df['ts_closed'].max()}")

    # Entry side distribution
    print(f"Long entries:  {(positions_df['entry_side'] == 'BUY').sum()}")
    print(f"Short entries: {(positions_df['entry_side'] == 'SELL').sum()}")

    # PnL summary
    pnl = positions_df["realized_pnl"].astype(float)
    winners = (pnl > 0).sum()
    losers = (pnl < 0).sum()
    print(f"Winners: {winners}  Losers: {losers}  Total PnL: {pnl.sum():,.2f}")

    # Required fields
    required = ["ts_opened", "ts_closed", "strategy_id", "instrument_id",
                "position_id", "entry_side", "peak_qty"]
    for col in required:
        nulls = positions_df[col].isna().sum()
        if nulls > 0:
            pos_issues.append(f"{col} has {nulls} NULLs")
            print(f"  ❌ {col}: {nulls} NULL values")

    # PnL fields should be populated for closed positions
    pnl_fields = ["avg_px_open", "avg_px_close", "realized_pnl", "realized_return", "currency"]
    for col in pnl_fields:
        nulls = positions_df[col].isna().sum()
        if nulls > 0:
            pos_issues.append(f"{col} has {nulls} NULLs in closed positions")
            print(f"  ⚠️  {col}: {nulls} NULLs (expected populated for closed positions)")

    # ts_opened must be before ts_closed
    bad_order = (positions_df["ts_opened"] >= positions_df["ts_closed"]).sum()
    if bad_order > 0:
        pos_issues.append(f"{bad_order} positions where ts_opened >= ts_closed")
        print(f"  ❌ {bad_order} positions where ts_opened >= ts_closed")

    # entry_side must be BUY or SELL
    valid_sides = {"BUY", "SELL"}
    bad_sides = positions_df[~positions_df["entry_side"].isin(valid_sides)]
    if len(bad_sides) > 0:
        pos_issues.append(f"{len(bad_sides)} positions with invalid entry_side")
        print(f"  ❌ Invalid entry_side values: {bad_sides['entry_side'].unique()}")

    # duration_ns should be positive
    bad_dur = (positions_df["duration_ns"].dropna().astype(float) <= 0).sum()
    if bad_dur > 0:
        pos_issues.append(f"{bad_dur} positions with zero/negative duration")
        print(f"  ⚠️  {bad_dur} positions with zero/negative duration")

    if not pos_issues:
        print("\n✅ Positions OK")
    else:
        print(f"\n⚠️  {len(pos_issues)} position issue(s)")

    print("\nSample positions:")
    display(positions_df.head(5))

In [ ]:
# ── Cell 6: Fill ↔ position cross-reference ──────────────────────
#
# Every closed position should have at least 2 fills (open + close).
# In NETTING mode, fills don't carry a position_id that maps cleanly
# to individual positions, so we check aggregate counts instead.

xref_issues = []

n_fills = len(fills_df)
n_positions = len(positions_df)

print(f"Fills:     {n_fills}")
print(f"Positions: {n_positions}")

if n_positions > 0 and n_fills > 0:
    # Each position needs at least an open + close fill.
    # Some fills may be partial, so fills >= 2 * positions is the minimum.
    min_expected_fills = n_positions * 2
    print(f"Minimum expected fills (2 per position): {min_expected_fills}")
    print(f"Fill/position ratio: {n_fills / n_positions:.1f}")

    if n_fills < min_expected_fills:
        xref_issues.append(
            f"Only {n_fills} fills for {n_positions} positions "
            f"(expected >= {min_expected_fills})"
        )
        print(f"  ⚠️  Fewer fills than expected — possible dropped events")

    # Time range sanity: first fill should be at or before first position open
    first_fill_ts = fills_df["ts"].min()
    first_pos_open = positions_df["ts_opened"].min()
    if first_fill_ts > first_pos_open:
        xref_issues.append("First fill is after first position open")
        print(f"  ⚠️  First fill ({first_fill_ts}) is after first position open ({first_pos_open})")

    # Last fill should be at or after last position close
    last_fill_ts = fills_df["ts"].max()
    last_pos_close = positions_df["ts_closed"].max()
    if last_fill_ts < last_pos_close:
        xref_issues.append("Last fill is before last position close")
        print(f"  ⚠️  Last fill ({last_fill_ts}) is before last position close ({last_pos_close})")

elif n_positions > 0 and n_fills == 0:
    xref_issues.append("Positions exist but no fills — PersistenceActor fill writing failed")
    print("❌ Positions exist but no fills — PersistenceActor fill writing failed")

elif n_positions == 0 and n_fills > 0:
    print("ℹ️  Fills exist but no closed positions — run may still have open positions")

if not xref_issues:
    print("\n✅ Fill ↔ position cross-reference OK")
else:
    print(f"\n⚠️  {len(xref_issues)} cross-reference issue(s)")

In [ ]:
# ── Cell 7: Account snapshots ─────────────────────────────────────

snapshots_df = pd.read_sql("""
    SELECT ts, venue, currency, balance_total, balance_free, balance_locked
    FROM account_snapshots
    WHERE run_id = %(run_id)s
    ORDER BY ts
""", engine, params={"run_id": RUN_ID})

snap_issues = []

print(f"Total snapshots: {len(snapshots_df)}")

if snapshots_df.empty:
    print("⚠️  No account snapshots — PersistenceActor timer may not have fired")
else:
    print(f"Date range: {snapshots_df['ts'].min()} → {snapshots_df['ts'].max()}")

    # Required fields
    for col in ["ts", "venue", "currency", "balance_total", "balance_free", "balance_locked"]:
        nulls = snapshots_df[col].isna().sum()
        if nulls > 0:
            snap_issues.append(f"{col} has {nulls} NULLs")
            print(f"  ❌ {col}: {nulls} NULL values")

    # Timestamps should be monotonically increasing
    if not snapshots_df["ts"].is_monotonic_increasing:
        snap_issues.append("Snapshot timestamps not monotonically increasing")
        print("  ⚠️  Snapshot timestamps are not monotonically increasing")

    # Snapshot interval check (expected every 60s per PersistenceActor)
    if len(snapshots_df) >= 2:
        diffs = snapshots_df["ts"].diff().dropna().dt.total_seconds()
        median_interval = diffs.median()
        print(f"Median interval: {median_interval:.0f}s (expected ~60s)")

        # Flag if interval is wildly off (> 2x expected)
        if median_interval > 130:
            snap_issues.append(f"Median snapshot interval is {median_interval:.0f}s (expected ~60s)")
            print(f"  ⚠️  Snapshot interval higher than expected")

        # Large gaps (> 5 minutes)
        big_gaps = diffs[diffs > 300]
        if len(big_gaps) > 0:
            print(f"  ⚠️  {len(big_gaps)} gaps > 5 minutes:")
            for idx in big_gaps.nlargest(5).index:
                print(f"       {snapshots_df['ts'].iloc[idx - 1]} → {snapshots_df['ts'].iloc[idx]}  ({big_gaps[idx]:.0f}s)")

    # Balance sanity: total should equal free + locked
    total = snapshots_df["balance_total"].astype(float)
    free = snapshots_df["balance_free"].astype(float)
    locked = snapshots_df["balance_locked"].astype(float)
    balance_mismatch = (abs(total - (free + locked)) > 0.01).sum()
    if balance_mismatch > 0:
        snap_issues.append(f"{balance_mismatch} snapshots where total != free + locked")
        print(f"  ❌ {balance_mismatch} snapshots where balance_total != balance_free + balance_locked")

    # Balance should never be negative
    neg_balance = (total < 0).sum()
    if neg_balance > 0:
        snap_issues.append(f"{neg_balance} snapshots with negative balance")
        print(f"  ❌ {neg_balance} snapshots with negative balance")

    if not snap_issues:
        print("\n✅ Account snapshots OK")
    else:
        print(f"\n⚠️  {len(snap_issues)} snapshot issue(s)")

    print(f"\nBalance at start: {total.iloc[0]:,.2f}")
    print(f"Balance at end:   {total.iloc[-1]:,.2f}")
    print(f"Change:           {total.iloc[-1] - total.iloc[0]:+,.2f}")

In [ ]:
# ── Cell 8: Data type verification ────────────────────────────────
#
# Financial columns must be NUMERIC in PostgreSQL, never FLOAT/DOUBLE.
# This checks the actual column types in the database schema.

type_df = pd.read_sql("""
    SELECT table_name, column_name, data_type, udt_name
    FROM information_schema.columns
    WHERE table_schema = 'public'
      AND table_name IN ('order_fills', 'positions', 'account_snapshots')
    ORDER BY table_name, ordinal_position
""", engine)

type_issues = []

# These columns MUST be numeric, never float/double
must_be_numeric = {
    "order_fills": ["last_qty", "last_px", "commission"],
    "positions": ["peak_qty", "avg_px_open", "avg_px_close",
                  "realized_pnl", "realized_return"],
    "account_snapshots": ["balance_total", "balance_free", "balance_locked"],
}

float_types = {"float4", "float8", "real", "double precision"}

for table, columns in must_be_numeric.items():
    table_types = type_df[type_df["table_name"] == table]
    for col in columns:
        col_row = table_types[table_types["column_name"] == col]
        if col_row.empty:
            type_issues.append(f"{table}.{col} — column not found")
            print(f"  ⚠️  {table}.{col} — column not found in schema")
            continue

        udt = col_row.iloc[0]["udt_name"]
        if udt in float_types:
            type_issues.append(f"{table}.{col} is {udt} — must be NUMERIC")
            print(f"  ❌ {table}.{col} is {udt} — MUST be NUMERIC")
        else:
            print(f"  ✅ {table}.{col} → {udt}")

if not type_issues:
    print("\n✅ All financial columns are NUMERIC")
else:
    print(f"\n❌ {len(type_issues)} type violation(s) — fix schema immediately")

In [ ]:
# ── Cell 9: Summary ───────────────────────────────────────────────

print("=" * 60)
print(f"  PERSISTENCE VERIFICATION SUMMARY")
print(f"  Run: {RUN_ID}")
print(f"  {run_row['strategy_id']} on {run_row['instrument_id']} ({run_row['run_mode']})")
print("=" * 60)

checks = []

# Lifecycle
if not lifecycle_issues:
    checks.append(("✅", "Run lifecycle", "Graceful shutdown, config present"))
else:
    checks.append(("⚠️", "Run lifecycle", "; ".join(lifecycle_issues)))

# Fills
if fills_df.empty:
    checks.append(("⚠️", "Order fills", "No fills found"))
elif not fill_issues:
    checks.append(("✅", "Order fills", f"{n_fills} fills, all valid"))
else:
    checks.append(("❌", "Order fills", "; ".join(fill_issues)))

# Positions
if positions_df.empty:
    checks.append(("⚠️", "Positions", "No closed positions"))
elif not pos_issues:
    checks.append(("✅", "Positions", f"{n_positions} positions, all valid"))
else:
    checks.append(("❌", "Positions", "; ".join(pos_issues)))

# Cross-reference
if not xref_issues:
    checks.append(("✅", "Fill ↔ position", "Counts and timestamps consistent"))
else:
    checks.append(("⚠️", "Fill ↔ position", "; ".join(xref_issues)))

# Snapshots
if snapshots_df.empty:
    checks.append(("⚠️", "Account snapshots", "No snapshots found"))
elif not snap_issues:
    checks.append(("✅", "Account snapshots", f"{len(snapshots_df)} snapshots, all valid"))
else:
    checks.append(("⚠️", "Account snapshots", "; ".join(snap_issues)))

# Data types
if not type_issues:
    checks.append(("✅", "Data types", "All financial columns NUMERIC"))
else:
    checks.append(("❌", "Data types", "; ".join(type_issues)))

for icon, label, detail in checks:
    print(f"  {icon} {label:<22} {detail}")

all_pass = all(icon == "✅" for icon, _, _ in checks)
has_errors = any(icon == "❌" for icon, _, _ in checks)
print()
if all_pass:
    print("  VERDICT: ✅ Persistence pipeline verified.")
elif has_errors:
    print("  VERDICT: ❌ Critical issues found — investigate before trusting persisted data.")
else:
    print("  VERDICT: ⚠️  Warnings present — review before relying on this data.")
print("=" * 60)